# 85 — Chỉ báo và dữ liệu ngoài giá

backtrader chỉ biết OHLCV. Mọi thứ khác — chỉ báo tính sẵn, dòng tiền khối ngoại,
P/E, ROE — phải đi vào bằng **line phụ** của feed (`bt_vn.feed_tu_finlens(..., cot_phu=(...))`).
Mỗi line phụ là một chỗ nối, và notebook này đo ba chỗ nối dễ sai nhất:

| Chỗ nối | Cạm bẫy đo được |
|---|---|
| **Chỉ báo** `bt.ind.*` vs `df.finlens.*` | line phụ phải **quy đơn vị** như OHLC (và `attrs` mất là hết biết đơn vị) · hai thư viện lệch nhau ở **giai đoạn warm-up** |
| **Dòng tiền khối ngoại** (`investor.flow`, VND) | `merge` làm mất `df.attrs` → feed từ chối đoán đơn vị |
| **P/E, ROE quý** (`financials.indicators`) | finlens **không có ngày công bố BCTC** → gắn số quý vào ngày cuối quý là **nhìn trước** · P/E phải tính trên **giá thô** · một lời gọi 100 mã **lặng lẽ mất 1/4 số mã** · EPS TTM có quý **không được cập nhật** |

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import math
import time
import warnings

import backtrader as bt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay, thanh_doi_mau
from finlens_examples import bt_vn
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


## 1 · Hai cách có chỉ báo trong backtrader

1. **`bt.ind.*`** — backtrader tự tính trong `__init__`, trên chính feed.
2. **`df.finlens.*`** — tính sẵn bằng TA-Lib trên frame finlens, rồi nạp thành line phụ.

Cách 2 tiện khi chỉ báo không có trong backtrader, hoặc khi muốn chỉ báo đã
"ấm" ngay từ phiên đầu backtest. Tải 6 năm, backtest trên 5 năm cuối — năm thừa
dùng ở phần 4.

In [2]:
MA = "HPG"
SO_NAM = 5

dai = bt_vn.bo_phien_chua_dong(client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=SO_NAM + 1)), client)
MOC = pd.Timestamp(lui_ngay(HOM_NAY, nam=SO_NAM))
gia = dai[dai["date"] >= MOC]
print(f"{MA}: {len(dai)} phiên tải về · {len(gia)} phiên backtest từ {gia['date'].min():%d/%m/%Y}")
print(f"df.attrs còn sau khi lọc dòng: {'finlens' in gia.attrs}")


def tinh_chi_bao(df: pd.DataFrame) -> pd.DataFrame:
    return df.finlens.rsi(14).finlens.sma(20).finlens.ema(20).finlens.macd(12, slowperiod=26, signalperiod=9)


COT_CB = ("rsi_14", "sma_20", "ema_20", "macd_12_26_9", "macdsignal_12_26_9")
CB = tinh_chi_bao(gia)
print("\nĐơn vị finlens ghi cho từng cột chỉ báo:")
for c in COT_CB:
    print(f"  {c:<20} {CB.attrs['finlens']['units'][c]}")

HPG: 1496 phiên tải về · 1245 phiên backtest từ 20/10/2021
df.attrs còn sau khi lọc dòng: True

Đơn vị finlens ghi cho từng cột chỉ báo:
  rsi_14               pct
  sma_20               kVND
  ema_20               kVND
  macd_12_26_9         kVND
  macdsignal_12_26_9   kVND


## 2 · ⚠️ Line phụ phải được quy đơn vị như OHLC

`bt_vn.chuan_bi_frame` nhân OHLC với 1.000 để về VND. Line phụ thì tuỳ: `rsi_14`
(đơn vị `pct`) phải giữ nguyên, còn `sma_20` mang đơn vị `kVND` phải nhân 1.000 như
giá. `bt_vn` đọc đơn vị **của từng line phụ** từ `df.attrs` để quyết định. Tham số
`don_vi_cot_phu={cot: None}` tắt việc quy đổi — dùng nó để xem điều gì xảy ra nếu
line phụ bị nạp nguyên kVND, trong một chiến lược "giá trên SMA20":

In [3]:
class DemTrenSMA(bt.Strategy):
    def __init__(self):
        self.sma_bt = bt.ind.SMA(self.data.close, period=20)
        self.dem = {"phien": 0, "bt.ind.SMA": 0, "line phụ sma_20": 0}

    def next(self):
        self.dem["phien"] += 1
        self.dem["bt.ind.SMA"] += self.data.close[0] > self.sma_bt[0]
        self.dem["line phụ sma_20"] += self.data.close[0] > self.data.sma_20[0]
        self.ty_so = self.data.close[0] / self.data.sma_20[0]


def dem_tren_sma(df: pd.DataFrame, **kw) -> None:
    ce = bt.Cerebro(stdstats=False)
    ce.adddata(bt_vn.feed_tu_finlens(df, cot_phu=("sma_20",), **kw))
    ce.addstrategy(DemTrenSMA)
    s = ce.run()[0]
    n = s.dem["phien"]
    print(f"  bt.ind.SMA {s.dem['bt.ind.SMA'] / n:.1%} · line phụ sma_20 {s.dem['line phụ sma_20'] / n:.1%} "
          f"· close / sma_20 phiên cuối = {s.ty_so:,.3f}   ({n} phiên)")


print("Phiên giá đóng cửa nằm trên SMA20:")
print("SAI — tắt quy đổi line phụ (don_vi_cot_phu={'sma_20': None}):")
dem_tren_sma(CB, don_vi_cot_phu={"sma_20": None})
print("ĐÚNG — mặc định, đơn vị đọc từ df.attrs:")
dem_tren_sma(CB)

Phiên giá đóng cửa nằm trên SMA20:
SAI — tắt quy đổi line phụ (don_vi_cot_phu={'sma_20': None}):


  bt.ind.SMA 48.2% · line phụ sma_20 100.0% · close / sma_20 phiên cuối = 1,002.072   (1226 phiên)
ĐÚNG — mặc định, đơn vị đọc từ df.attrs:


  bt.ind.SMA 48.2% · line phụ sma_20 48.3% · close / sma_20 phiên cuối = 1.002   (1226 phiên)

Nạp nguyên kVND thì tín hiệu "giá trên SMA" bật **mọi phiên** — không lỗi, không
cảnh báo, chỉ là một chiến lược khác hẳn. Mặc định của `bt_vn` cho đúng con số
của `bt.ind.SMA`.

Chỗ còn hở: đơn vị line phụ đọc từ `attrs`, mà `attrs` **mất sau `merge`/`concat`**.
Khi đó `don_vi_gia` bị bắt buộc truyền (hàm từ chối đoán), nhưng line phụ không có
đơn vị thì bị coi là "không cần quy đổi":

In [4]:
ngoai = pd.DataFrame({"date": CB["date"], "cot_khac": 0.0})  # frame không phải của finlens, vd dòng tiền đã xử lý
mat_attrs = CB.merge(ngoai, on="date")
print(f"df.attrs sau merge với một frame không có attrs: {mat_attrs.attrs}")
print("Chỉ truyền don_vi_gia='kVND':")
with warnings.catch_warnings(record=True) as canh_bao:
    warnings.simplefilter("always")
    dem_tren_sma(mat_attrs, don_vi_gia="kVND")
for w in canh_bao:
    print(f"  ⚠️ {w.category.__name__}: {w.message}")
print("Truyền thêm don_vi_cot_phu={'sma_20': 'kVND'}:")
dem_tren_sma(mat_attrs, don_vi_gia="kVND", don_vi_cot_phu={"sma_20": "kVND"})

df.attrs sau merge với một frame không có attrs: {}
Chỉ truyền don_vi_gia='kVND':
  bt.ind.SMA 48.2% · line phụ sma_20 100.0% · close / sma_20 phiên cuối = 1,002.072   (1226 phiên)
  ⚠️ UserWarning: Line phụ ['sma_20'] không có đơn vị (frame đã mất attrs) nên được giữ nguyên. Nếu là cột giá kVND, truyền don_vi_cot_phu={cot: 'kVND'}; nếu không phải cột giá, truyền {cot: None} để tắt cảnh báo này.
Truyền thêm don_vi_cot_phu={'sma_20': 'kVND'}:


  bt.ind.SMA 48.2% · line phụ sma_20 48.3% · close / sma_20 phiên cuối = 1.002   (1226 phiên)


Không có lỗi, nhưng `bt_vn` giờ **phát `UserWarning`** (in ở trên) khi frame đã mất
`attrs` mà còn line phụ chưa khai đơn vị. Cách xử lý: khai đơn vị line phụ **tường
minh** như khai `don_vi_gia` — `don_vi_cot_phu={"sma_20": "kVND"}` cho cột giá, hoặc
`{cot: None}` cho cột không phải giá (như các tỷ lệ ở phần 5–6). Từ
đây trở đi, mọi feed có chỉ báo đều nạp từ frame còn nguyên `attrs` (kiểm bằng
dòng in dưới):

In [5]:
print("Đơn vị bt_vn sẽ đọc cho từng line phụ:", {c: CB.attrs["finlens"]["units"][c] for c in COT_CB})

Đơn vị bt_vn sẽ đọc cho từng line phụ: {'rsi_14': 'pct', 'sma_20': 'kVND', 'ema_20': 'kVND', 'macd_12_26_9': 'kVND', 'macdsignal_12_26_9': 'kVND'}


## 3 · Đối chiếu RSI / SMA / EMA / MACD — cùng một cửa sổ dữ liệu

Cả hai thư viện nhận **cùng 5 năm giá**. Một strategy rỗng chỉ để backtrader tính
chỉ báo; sau khi chạy, đọc toàn bộ mảng của từng line ra pandas.

In [6]:
class GhiChiBao(bt.Strategy):
    def __init__(self):
        self.rsi = bt.ind.RSI(self.data.close, period=14)
        self.sma = bt.ind.SMA(self.data.close, period=20)
        self.ema = bt.ind.EMA(self.data.close, period=20)
        self.macd = bt.ind.MACD(self.data.close, period_me1=12, period_me2=26, period_signal=9)


def doi_chieu(df_cb: pd.DataFrame) -> pd.DataFrame:
    ce = bt.Cerebro(stdstats=False)
    ce.adddata(bt_vn.feed_tu_finlens(df_cb, cot_phu=COT_CB))  # bt_vn quy sma/ema/macd kVND → VND
    ce.addstrategy(GhiChiBao)
    s = ce.run()[0]
    d = s.data
    return pd.DataFrame(
        {
            "rsi_bt": s.rsi.array, "rsi_fl": d.rsi_14.array,
            "sma_bt": s.sma.array, "sma_fl": d.sma_20.array,
            "ema_bt": s.ema.array, "ema_fl": d.ema_20.array,
            "macd_bt": s.macd.macd.array, "macd_fl": d.macd_12_26_9.array,
            "tin_hieu_bt": s.macd.signal.array, "tin_hieu_fl": d.macdsignal_12_26_9.array,
        },
        index=pd.DatetimeIndex([bt.num2date(x) for x in d.datetime.array], name="date"),
    )


CAP = [("RSI 14", "rsi", "điểm RSI"), ("SMA 20", "sma", "VND"), ("EMA 20", "ema", "VND"),
       ("MACD", "macd", "VND"), ("Tín hiệu MACD", "tin_hieu", "VND")]
WARM = 100  # phiên đầu tiên kể từ khi CẢ HAI thư viện có giá trị


def bang_lech(dc: pd.DataFrame) -> pd.DataFrame:
    hang = []
    for ten, k, dv in CAP:
        a, b = dc[f"{k}_bt"], dc[f"{k}_fl"]
        lech = (a - b).abs().dropna()
        hang.append(
            {
                "chỉ báo": ten,
                "phiên đầu có giá trị — backtrader": dc.index.get_loc(a.first_valid_index()) + 1,
                "— finlens": dc.index.get_loc(b.first_valid_index()) + 1,
                f"lệch lớn nhất, {WARM} phiên đầu": lech.iloc[:WARM].max(),
                "lệch lớn nhất, sau đó": lech.iloc[WARM:].max(),
                "đơn vị": dv,
            }
        )
    return pd.DataFrame(hang)


dc = doi_chieu(CB)
bang_lech(dc)

,chỉ báo,phiên đầu có giá trị — backtrader,— finlens,"lệch lớn nhất, 100 phiên đầu","lệch lớn nhất, sau đó",đơn vị
0,RSI 14,15,15,4.263256e-14,9.947598e-14,điểm RSI
1,SMA 20,20,20,1.091394e-11,2.546585e-11,VND
2,EMA 20,20,20,1.455192e-11,1.455192e-11,VND
3,MACD,26,34,7.495183e+01,4.165965e-06,VND
4,Tín hiệu MACD,34,34,1.601919e+02,1.525185e-05,VND


Đọc bảng:

- **RSI, SMA, EMA khớp tới sai số dấu phẩy động** ngay từ phiên đầu. backtrader và
  TA-Lib khởi tạo ba chỉ báo này giống nhau: RSI làm trơn kiểu Wilder bắt đầu bằng
  trung bình cộng 14 phiên, EMA bắt đầu bằng SMA của `period` phiên đầu.
- **MACD thì không.** TA-Lib chỉ trả MACD khi đường tín hiệu đã đủ 9 phiên, và khởi
  tạo EMA nhanh muộn để thẳng hàng với EMA chậm; backtrader trả MACD sớm hơn và EMA
  nhanh của nó đã chạy từ phiên 12. Hai điểm xuất phát khác nhau, rồi **hội tụ**
  vì EMA quên dần điểm xuất phát — sau 100 phiên chỉ còn lệch dưới một đồng.

In [7]:
lech_macd = pd.concat(
    [
        pd.DataFrame({"phien": range(1, len(dc) + 1), "lech": (dc["macd_bt"] - dc["macd_fl"]).abs().values, "duong": "MACD"}),
        pd.DataFrame({"phien": range(1, len(dc) + 1), "lech": (dc["tin_hieu_bt"] - dc["tin_hieu_fl"]).abs().values, "duong": "Tín hiệu MACD"}),
    ]
).dropna()
fig = duong(
    lech_macd[lech_macd["phien"] <= 200],
    x="phien",
    y="lech",
    theo="duong",
    tieu_de="MACD: |backtrader − finlens| theo số phiên kể từ đầu feed",
    phu_de="Thang log — lệch giảm theo cấp số nhân, đúng như một EMA quên dần điểm khởi tạo",
    nhan_y="VND (thang log)",
)
fig.update_yaxes(type="log")
fig.update_xaxes(title_text="phiên thứ")
fig

## 4 · ⚠️ Warm-up theo độ dài lịch sử

Phần 3 cho hai thư viện **cùng dữ liệu đầu vào**. Thực tế hay gặp hơn là: tính
chỉ báo bằng finlens trên **toàn bộ lịch sử tải về** (6 năm), rồi cắt lấy cửa sổ
backtest (5 năm). Line phụ khi đó đã "ấm" từ phiên đầu, còn `bt.ind.*` bắt đầu
lạnh. Cùng công thức, khác điểm xuất phát:

In [8]:
CB_DAI = tinh_chi_bao(dai)
CB_DAI = CB_DAI[CB_DAI["date"] >= MOC]  # lọc dòng giữ attrs
dc_dai = doi_chieu(CB_DAI)

khoi = []
for dau in range(0, 200, 20):
    hang = {"phiên": f"{dau + 1}–{dau + 20}"}
    for ten, k, dv in CAP[:1] + CAP[2:4]:
        hang[f"{ten} ({dv})"] = (dc_dai[f"{k}_bt"] - dc_dai[f"{k}_fl"]).abs().iloc[dau : dau + 20].max()
    khoi.append(hang)
bang_khoi = pd.DataFrame(khoi)

lech_rsi = (dc_dai["rsi_bt"] - dc_dai["rsi_fl"]).abs()
vuot = lech_rsi[lech_rsi >= 0.01]
print(f"RSI: lệch lớn nhất {lech_rsi.max():.2f} điểm; từ phiên thứ {dc_dai.index.get_loc(vuot.index[-1]) + 2} trở đi "
      f"luôn dưới 0,01 điểm")
bang_khoi.round(4)

RSI: lệch lớn nhất 6.76 điểm; từ phiên thứ 98 trở đi luôn dưới 0,01 điểm


,phiên,RSI 14 (điểm RSI),EMA 20 (VND),MACD (VND)
0,1–20,6.7617,333.4122,NaN
1,21–40,2.0876,301.6587,691.9591
2,41–60,0.6878,40.7570,220.3858
3,61–80,0.2582,5.5067,47.4243
4,81–100,0.0462,0.7440,10.1798
5,101–120,0.0081,0.1005,2.1842
6,121–140,0.0022,0.0136,0.4686
7,141–160,0.0003,0.0018,0.1005
8,161–180,0.0001,0.0002,0.0216
9,181–200,0.0000,0.0000,0.0046


In [9]:
ve = pd.DataFrame({"phien": range(1, len(dc_dai) + 1), "lech": lech_rsi.values}).dropna()
fig = duong(
    ve[ve["phien"] <= 200],
    x="phien",
    y="lech",
    tieu_de="RSI 14: line phụ tính trên 6 năm vs bt.ind.RSI tính trên 5 năm",
    phu_de="Cùng công thức Wilder — lệch chỉ vì điểm bắt đầu làm trơn khác nhau",
    nhan_y="điểm RSI",
)
fig.update_xaxes(title_text="phiên thứ (kể từ đầu cửa sổ backtest)")
fig

Một chiến lược "mua khi RSI < 30" chạy trên hai nguồn này sẽ ra **tín hiệu khác
nhau trong vài tháng đầu** — dù không thư viện nào sai. Hai cách xử lý, chọn một
và ghi rõ:

1. Nạp feed **dài hơn** cửa sổ backtest và chỉ cho giao dịch từ `MOC` trở đi
   (`bt.ind.*` có thời gian ấm lên).
2. Tính bằng finlens trên lịch sử dài rồi nạp line phụ — nhưng khi đó chỉ báo
   trong backtest phải được **tính lại đúng như vậy** khi chạy thật mỗi ngày.

## 5 · Dòng tiền khối ngoại làm line phụ

In [10]:
dong_tien_tho = client.eod.stock.investor.flow(MA, start=lui_ngay(HOM_NAY, nam=SO_NAM))
print(f"Nhóm nhà đầu tư: {sorted(dong_tien_tho['group'].unique())}")
print(f"Đơn vị: net_value {dong_tien_tho.attrs['finlens']['units']['net_value']} · "
      f"net_volume {dong_tien_tho.attrs['finlens']['units']['net_volume']}")

thua = sorted(set(dong_tien_tho["date"]) - set(gia["date"]))
print(f"\nPhiên có ở dòng tiền mà không có ở giá: {[f'{d:%d/%m/%Y}' for d in thua]}")
dong_tien = bt_vn.bo_phien_chua_dong(dong_tien_tho, client)
print(f"Sau bo_phien_chua_dong: còn {len(set(dong_tien['date']) - set(gia['date']))} phiên thừa · "
      f"thiếu {len(set(gia['date']) - set(dong_tien['date']))} phiên so với giá")

Nhóm nhà đầu tư: ['foreign']
Đơn vị: net_value VND · net_volume share

Phiên có ở dòng tiền mà không có ở giá: ['20/10/2026']
Sau bo_phien_chua_dong: còn 0 phiên thừa · thiếu 0 phiên so với giá


Lúc chạy notebook này (trong giờ giao dịch), `investor.flow` cũng trả về **phiên
đang chạy** như `ohlcv` — số khối ngoại mua/bán tính tới lúc gọi. Phải bỏ nó cùng
lúc với thanh giá dở dang; `bt_vn.bo_phien_chua_dong` dùng được cho mọi frame có
cột `date`.

Ghép vào frame giá bằng `merge` — và đây là chỗ `df.attrs` biến mất:

In [11]:
nn = dong_tien[dong_tien["group"] == "foreign"][["date", "net_value"]]
khung = gia.merge(nn, on="date", how="left")
print(f"df.attrs sau merge: {khung.attrs}")

khung["gtgd"] = khung["close"] * 1_000 * khung["volume"]  # VND — close là kVND
khung["nn_ty_le_20"] = khung["net_value"].rolling(20).sum() / khung["gtgd"].rolling(20).sum()
print(f"Phiên thiếu số khối ngoại: {khung['net_value'].isna().sum()}")

try:
    bt_vn.feed_tu_finlens(khung, cot_phu=("nn_ty_le_20",))
except ValueError as e:
    print(f"\nbt_vn từ chối: {e}")

feed_nn = bt_vn.feed_tu_finlens(
    khung, don_vi_gia="kVND", cot_phu=("nn_ty_le_20",),
    don_vi_cot_phu={"nn_ty_le_20": None},  # tỷ lệ không đơn vị — không quy đổi
)
print("→ Khai don_vi_gia='kVND' và đơn vị line phụ tường minh thì chạy.")

df.attrs sau merge: {}
Phiên thiếu số khối ngoại: 0

bt_vn từ chối: Frame không còn df.attrs['finlens'] (thường do pd.concat/merge). Truyền don_vi_gia='kVND' | 'VND' | 'index_point' cho rõ.
→ Khai don_vi_gia='kVND' và đơn vị line phụ tường minh thì chạy.


`nn_ty_le_20` = khối ngoại mua ròng 20 phiên ÷ giá trị giao dịch 20 phiên: tỷ lệ
không đơn vị, so được giữa các mã. Cả tử và mẫu đều là VND — nếu quên nhân
`close` (kVND) với 1.000, tỷ lệ phình 1.000 lần; dấu vẫn đúng nên tín hiệu `> 0`
không đổi, nhưng mọi ngưỡng khác 0 sẽ sai.

### Dòng tiền khối ngoại có cải thiện một chiến lược xu hướng không?

Hai bản, khác đúng một điều kiện: **xu hướng** (giá trên SMA50) so với **xu hướng
và khối ngoại mua ròng 20 phiên**. Chạy trên 10 mã doanh nghiệp phi tài chính
thanh khoản nhất HOSE — một mã thì quá ít để kết luận.

In [12]:
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
MA_CT = danh_muc[danh_muc["company_type"] == "CT"]["symbol"].tolist()
g3t = client.eod.stock.ohlcv(MA_CT, start=lui_ngay(HOM_NAY, thang=3))
assert not g3t.attrs["finlens"]["truncated"]
gtgd_bq = (g3t["close"] * 1_000 * g3t["volume"]).groupby(g3t["symbol"]).mean()
VU_TRU = gtgd_bq.nlargest(100).index.tolist()
print(f"{len(MA_CT)} mã CT trên HOSE → vũ trụ 100 mã GTGD bình quân 3 tháng lớn nhất "
      f"(thấp nhất {gtgd_bq.nlargest(100).min() / 1e9:,.1f} tỷ/phiên)")
print("10 mã đầu:", VU_TRU[:10])


def tai_gia(ma_ds: list[str], *, nam: float, adjusted: bool = True) -> pd.DataFrame:
    """Tải theo lô 50 mã, kiểm truncated từng lô (xem notebook 34)."""
    phan = []
    for i in range(0, len(ma_ds), 50):
        d = client.eod.stock.ohlcv(ma_ds[i : i + 50], start=lui_ngay(HOM_NAY, nam=nam), adjusted=adjusted)
        assert not d.attrs["finlens"]["truncated"], "lô bị cắt — giảm cỡ lô"
        assert d.attrs["finlens"]["units"]["close"] == "kVND"
        phan.append(d)
    return bt_vn.bo_phien_chua_dong(pd.concat(phan, ignore_index=True), client)  # attrs mất ở concat


t0 = time.perf_counter()
GIA = tai_gia(VU_TRU, nam=7)
print(f"Giá điều chỉnh 7 năm: {len(GIA):,} dòng · {GIA['symbol'].nunique()} mã · {time.perf_counter() - t0:.0f} giây")

360 mã CT trên HOSE → vũ trụ 100 mã GTGD bình quân 3 tháng lớn nhất (thấp nhất 5.0 tỷ/phiên)
10 mã đầu: ['VIC', 'VHM', 'FPT', 'HPG', 'MSN', 'MWG', 'BSR', 'NVL', 'PNJ', 'GEX']


Giá điều chỉnh 7 năm: 164,704 dòng · 100 mã · 41 giây


In [13]:
MA_NN = VU_TRU[:10]
dt10 = bt_vn.bo_phien_chua_dong(client.eod.stock.investor.flow(MA_NN, start=lui_ngay(HOM_NAY, nam=SO_NAM)), client)
assert not dt10.attrs["finlens"]["truncated"]
dt10 = dt10[dt10["group"] == "foreign"][["symbol", "date", "net_value"]]


class XuHuongNN(bt_vn.ChienLuocVN):
    params = (("dung_nn", False), ("sma", 50))

    def __init__(self):
        super().__init__()
        self.sma = bt.ind.SMA(self.data.close, period=self.p.sma)
        self.lenh = None

    def notify_order(self, order):
        super().notify_order(order)
        if order.status in (order.Completed, order.Canceled, order.Margin, order.Rejected, order.Expired):
            self.lenh = None

    def dieu_kien(self) -> bool:
        tot = self.data.close[0] > self.sma[0]
        if self.p.dung_nn:
            tot = tot and self.data.nn_ty_le_20[0] > 0
        return tot

    def next(self):
        if self.lenh:
            return
        if not self.getposition().size:
            if self.dieu_kien():
                self.lenh = self.buy()
        elif not self.dieu_kien():
            self.lenh = self.ban()


def chay(frame: pd.DataFrame, lop, cot_phu: tuple[str, ...], **tham_so) -> dict:
    ce = bt.Cerebro(stdstats=False)
    # Frame đã qua merge nên mất attrs: khai cả đơn vị giá lẫn đơn vị line phụ. Mọi line phụ dùng
    # ở đây đều không phải giá (tỷ lệ, LNST VND, chênh ROE, P/E) → None = giữ nguyên.
    ce.adddata(bt_vn.feed_tu_finlens(frame, don_vi_gia="kVND", cot_phu=cot_phu,
                                     don_vi_cot_phu={c: None for c in cot_phu}))
    ce.broker.setcash(100_000_000)
    ce.broker.addcommissioninfo(bt_vn.PhiCoPhieuVN())
    ce.broker.set_filler(bt_vn.KhopTheoBienDo("HOSE"))
    ce.addsizer(bt_vn.LoVN)
    ce.addstrategy(lop, **tham_so)
    bt_vn.them_analyzer(ce)
    return bt_vn.ket_qua(ce.run()[0])


hang = []
for ma in MA_NN:
    f = GIA[(GIA["symbol"] == ma) & (GIA["date"] >= MOC)].merge(dt10[dt10["symbol"] == ma].drop(columns="symbol"), on="date", how="left")
    f["nn_ty_le_20"] = f["net_value"].rolling(20).sum() / (f["close"] * 1_000 * f["volume"]).rolling(20).sum()
    for ten, dung_nn in [("xu hướng", False), ("xu hướng + khối ngoại", True)]:
        t = chay(f, XuHuongNN, ("nn_ty_le_20",), dung_nn=dung_nn)["tom_tat"]
        hang.append({"mã": ma, "bản": ten, "CAGR %": t["cagr"] * 100, "MDD %": t["mdd"] * 100,
                     "Sharpe (rf=0)": t["sharpe"], "số vòng": t["so_vong_dong"]})

kq_nn = pd.DataFrame(hang)
bang_nn = kq_nn.pivot(index="mã", columns="bản", values=["CAGR %", "MDD %", "số vòng"]).round(2)
bang_nn.loc["trung bình"] = bang_nn.mean().round(2)
bang_nn

CAGR %                          MDD %                        \
bản        xu hướng xu hướng + khối ngoại xu hướng xu hướng + khối ngoại   
mã                                                                         
BSR           11.83                 -2.14   -41.35                -51.73   
FPT           -0.41                 -4.97   -42.63                -28.34   
GEX           11.36                 -6.87   -52.27                -41.06   
HPG           -2.81                  2.62   -33.34                -23.71   
MSN           -9.06                 -8.66   -41.27                -40.82   
MWG            2.71                 -0.37   -35.81                -21.59   
NVL           -3.61                  2.67   -47.45                -31.72   
PNJ           -2.31                 -6.55   -36.25                -41.09   
VHM           12.57                 -0.88   -31.27                -24.24   
VIC           28.94                  1.22   -44.10                -43.02   
trung bình     4.92                 -2.39   -40.57                -34.73   

            số vòng                        
bản        xu hướng xu hướng + khối ngoại  
mã                                         
BSR            32.0                  46.0  
FPT            45.0                  38.0  
GEX            36.0                  33.0  
HPG            40.0                  31.0  
MSN            40.0                  32.0  
MWG            37.0                  36.0  
NVL            27.0                  25.0  
PNJ            38.0                  56.0  
VHM            27.0                  16.0  
VIC            35.0                  23.0  
trung bình     35.7                  33.6

Thêm điều kiện khối ngoại **không** cải thiện chiến lược xu hướng trên 10 mã này:
CAGR trung bình giảm từ **+4,92%** xuống **−2,39%/năm**, chỉ 3/10 mã (HPG, MSN, NVL)
khá hơn. Đổi lại, sụt sâu nhất trung bình nông hơn (−34,73% so với −40,57%). Cả
hai bản đều quay ~34–36 vòng trong 5 năm — SMA50 trên cổ phiếu Việt Nam bị nhiễu
nhiều, và mỗi vòng tốn ít nhất 0,4% phí + thuế.

Điểm của mục này không phải "khối ngoại vô dụng". Điểm là line phụ cho phép thử giả
thuyết đó bằng **đúng bộ máy** khớp lệnh, phí, T+2, trần/sàn như mọi chiến lược
khác — và kết quả được phép là âm.

## 6 · P/E và ROE quý — ⚠️ thiên lệch nhìn trước

### 6.1 Chọn đúng loại hình doanh nghiệp

ROE quý của doanh nghiệp `CT` (phi tài chính) là **riêng quý đó**, còn của `NH`,
`CK`, `BH` đã **quy về năm** (notebook [`22`](../02-phan-tich-co-ban/22_chi_so_va_dupont.ipynb)).
Trộn hai loại thì mọi phép so ROE vô nghĩa — nên vũ trụ ở trên chỉ gồm `CT`:

In [14]:
mau = client.financials.indicators(["FPT", "VCB"], codes=["roe"], period="quarterly", start_year=HOM_NAY.year - 2)
print(mau.groupby(["symbol", "company_type"], observed=True)["value"].median().rename("ROE quý trung vị").to_string())

MA_CHI_SO = ["eps", "roe", "pe", "npat"]
NAM_DAU = HOM_NAY.year - 9

# Cách ngây thơ: một lời gọi cho cả 100 mã — max_symbols_per_request = 100 nên "được phép"
t0 = time.perf_counter()
mot_lan = client.financials.indicators(VU_TRU, codes=MA_CHI_SO, period="quarterly", start_year=NAM_DAU)
m = mot_lan.attrs["finlens"]
thieu = sorted(set(VU_TRU) - set(mot_lan["symbol"]))
print(f"Một lời gọi: {len(mot_lan):,} dòng · {mot_lan['symbol'].nunique()}/{len(VU_TRU)} mã · truncated = {m['truncated']} · "
      f"{time.perf_counter() - t0:.0f} giây")
print(f"  cảnh báo trong attrs: {[w['code'] + ' ' + str(w['detail']) for w in m['warnings']]}")
print(f"  mã vắng mặt ({len(thieu)}): {', '.join(thieu)}")
print(f"  mã cuối cùng nhận được theo chữ cái: {max(mot_lan['symbol'])} · mã vắng đầu tiên: {thieu[0]}")

symbol  company_type
FPT     CT              0.057327
VCB     NH              0.179825


Một lời gọi: 10,636 dòng · 75/100 mã · truncated = False · 19 giây
  cảnh báo trong attrs: ["PERIOD_DROPPED_INCOMPLETE {'periods': [{'symbol': 'PVD', 'year': 2022, 'quarter': 2}]}"]
  mã vắng mặt (25): PVP, PVT, REE, SAB, SBT, SZC, TAL, TCH, TCM, TDP, TLG, VCG, VGC, VHC, VHM, VIC, VJC, VNM, VOS, VPI, VPL, VRE, VSC, VTP, VVS
  mã cuối cùng nhận được theo chữ cái: PVD · mã vắng đầu tiên: PVP


C:\Program Files\Python312\Lib\asyncio\events.py:88: DataQualityWarning: [PERIOD_DROPPED_INCOMPLETE] Hạn mức số dòng cắt ngang các kỳ sau nên chúng bị **loại khỏi kết quả**: PVD (Q2 2022). Một kỳ nạp thiếu khoản mục không cho ra ô trống mà cho ra con số nhỏ hơn sự thật, nên phát nó ra còn tệ hơn bỏ đi. Thu hẹp `start_year`/`end_year` hoặc số mã để lấy đủ.
  self._context.run(self._callback, *self._args)


⚠️ **Một lời gọi 100 mã mất 25 mã — và `truncated = False`.** Hạn mức số dòng cắt
ngang kết quả. Thư viện bỏ kỳ dở dang và cảnh báo đúng một kỳ (PVD Q2/2022),
nhưng **mọi mã đứng sau PVD theo thứ tự chữ cái** — từ PVP tới VVS, gồm cả VIC, VHM,
VNM — biến mất mà cảnh báo không nhắc tên mã nào. `max_symbols_per_request = 100`
nói lời gọi hợp lệ; nó không nói lời gọi trả đủ.

Cách phòng: tải theo lô nhỏ, coi mọi phần tử trong `attrs['finlens']['warnings']`
là lỗi, và **đếm số mã nhận về**.

In [15]:
def tai_chi_so(ma_ds: list[str], *, lo: int = 20) -> pd.DataFrame:
    """Tải chỉ số quý theo lô nhỏ; dừng nếu lô nào bị cắt hoặc bị bỏ kỳ."""
    phan = []
    for i in range(0, len(ma_ds), lo):
        d = client.financials.indicators(ma_ds[i : i + lo], codes=MA_CHI_SO, period="quarterly", start_year=NAM_DAU)
        m = d.attrs["finlens"]
        assert not m["truncated"] and not m["warnings"], f"lô {i // lo} bị cắt: {m['warnings']}"
        phan.append(d)
    return pd.concat(phan, ignore_index=True)


t0 = time.perf_counter()
CS = tai_chi_so(VU_TRU)
print(f"Theo lô 20 mã: {len(CS):,} dòng · {CS['symbol'].nunique()}/{len(VU_TRU)} mã · loại hình {sorted(CS['company_type'].unique())} · "
      f"{time.perf_counter() - t0:.0f} giây")
print("Đơn vị (cột `unit`, theo từng dòng):", {k: list(v) for k, v in CS.groupby("code", observed=True)["unit"].unique().items()})
print("Cột ngày công bố trong frame:", [c for c in CS.columns if "date" in c or "publish" in c or "release" in c] or "KHÔNG CÓ")

Theo lô 20 mã: 14,208 dòng · 100/100 mã · loại hình ['CT'] · 39 giây
Đơn vị (cột `unit`, theo từng dòng): {'eps': ['VND/share'], 'npat': ['VND'], 'pe': ['x'], 'roe': ['ratio']}
Cột ngày công bố trong frame: KHÔNG CÓ


### 6.2 Bằng chứng: số quý không có mặt vào ngày cuối quý

In [16]:
moi_nhat = CS.assign(ky=CS["year"].astype(int) * 10 + CS["quarter"].astype(int)).groupby("symbol")["ky"].max()
quy_hien_tai = pd.Period(HOM_NAY, freq="Q")
cuoi_quy_truoc = (quy_hien_tai - 1).end_time.normalize()
print(f"Hôm nay {HOM_NAY:%d/%m/%Y} — đã {(pd.Timestamp(HOM_NAY) - cuoi_quy_truoc).days} ngày kể từ cuối {quy_hien_tai - 1}")
print("Quý mới nhất finlens có, đếm theo mã:")
print(moi_nhat.map(lambda k: f"Q{k % 10}/{k // 10}").value_counts().to_string())

Hôm nay 20/10/2026 — đã 20 ngày kể từ cuối 2026Q3
Quý mới nhất finlens có, đếm theo mã:
ky
Q2/2026    97
Q3/2026     3


20 ngày sau khi quý 3 kết thúc — đúng hạn BCTC quý riêng lẻ — finlens mới có quý 3
của **3/100** mã; 97 mã vẫn dừng ở quý 2. Một backtest gắn số quý vào ngày cuối quý
là giả định cả 100 mã đã công bố ngay ngày 30/9. Không ai biết những con số đó vào
ngày ấy.

### 6.3 Hạn công bố theo quy định

Thông tư **96/2020/TT-BTC** (hướng dẫn công bố thông tin trên thị trường chứng
khoán), Điều 14 — công bố thông tin định kỳ của tổ chức niêm yết:

| Báo cáo | Hạn | Công ty mẹ (lập BCTC hợp nhất) |
|---|---|---|
| BCTC quý (khoản 3) | 20 ngày kể từ cuối quý | **30 ngày** |
| BCTC bán niên đã soát xét (khoản 2) | 5 ngày từ khi ký soát xét, không quá 45 ngày kể từ cuối 6 tháng | **60 ngày** |
| BCTC năm đã kiểm toán (khoản 1, dẫn Điều 10) | 10 ngày từ khi ký kiểm toán, **không quá 90 ngày** kể từ cuối năm | 90 ngày |

finlens không nói số của quý 2 và quý 4 lấy từ BCTC quý hay từ bản soát xét / kiểm
toán (bản sau có thể điều chỉnh số), nên chọn **hạn của văn bản muộn nhất** phủ
kỳ đó — hạn bảo thủ:

| Quý | Độ trễ áp dụng |
|---|---|
| Q1, Q3 | 30 ngày |
| Q2 | 60 ngày |
| Q4 | 90 ngày |

Số quý chỉ được dùng từ **phiên đầu tiên sau** ngày hết hạn (báo cáo có thể ra
sau giờ đóng cửa của chính ngày đó). Hạn trên đối chiếu qua bản trích dẫn văn bản;
kiểm lại với văn bản hợp nhất mới nhất trước khi dùng cho tiền thật.

In [17]:
DO_TRE = {1: 30, 2: 60, 3: 30, 4: 90}  # ngày lịch, kể từ cuối quý

rong = CS.pivot_table(index=["symbol", "year", "quarter"], columns="code", values="value", observed=True).reset_index()
rong["ky"] = rong["year"].astype(int) * 4 + rong["quarter"].astype(int) - 1

phan = []
for ma, d in rong.groupby("symbol", observed=True):
    d = d.set_index("ky").reindex(range(int(d["ky"].min()), int(d["ky"].max()) + 1))  # quý thiếu → NaN, không trượt
    d["symbol"] = ma
    d["eps_khong_doi"] = d["eps"].diff() == 0  # EPS TTM y hệt quý trước — để kiểm dữ liệu
    d["lnst_tang"] = d["npat"] - d["npat"].shift(4)  # LNST quý so với cùng quý năm trước, VND
    d["lnst_tang_pct"] = d["lnst_tang"] / d["npat"].shift(4).abs()
    d["roe_yoy"] = d["roe"] - d["roe"].shift(4)  # ROE quý (CT: riêng quý) so với cùng quý năm trước
    phan.append(d)
QUY = pd.concat(phan).reset_index().dropna(subset=["year"])
QUY["year"] = QUY["year"].astype(int)
QUY["quarter"] = QUY["quarter"].astype(int)
QUY["symbol"] = QUY["symbol"].astype(GIA["symbol"].dtype)

kieu_ngay = GIA["date"].dtype  # merge_asof của pandas 3 đòi cùng độ phân giải thời gian
QUY["cuoi_quy"] = pd.to_datetime(
    [pd.Period(year=y, quarter=q, freq="Q").end_time.normalize() for y, q in zip(QUY["year"], QUY["quarter"])]
).astype(kieu_ngay)
QUY["han_cong_bo"] = (QUY["cuoi_quy"] + pd.to_timedelta(QUY["quarter"].map(DO_TRE), unit="D")).astype(kieu_ngay)
QUY[["symbol", "year", "quarter", "eps", "npat", "roe", "lnst_tang_pct", "roe_yoy", "cuoi_quy", "han_cong_bo"]].tail(4)

code,symbol,year,quarter,eps,npat,roe,lnst_tang_pct,roe_yoy,cuoi_quy,han_cong_bo
3535,VVS,2025,3,8219.096227,8.894353e+10,0.177428,1.106826,0.055524,2025-09-30,2025-10-30
3536,VVS,2025,4,15160.289816,1.560393e+11,0.237383,22.534801,0.218598,2025-12-31,2026-03-31
3537,VVS,2026,1,15160.289816,1.664841e+11,0.203362,5.389140,0.134535,2026-03-31,2026-04-30
3538,VVS,2026,2,10842.091034,2.455816e+11,0.230764,3.442102,0.103342,2026-06-30,2026-08-29


### ⚠️ Kiểm dữ liệu trước khi chọn tín hiệu: EPS TTM có được cập nhật mỗi quý không?

Tín hiệu tự nhiên nhất là "EPS TTM tăng so với quý trước". Nhưng EPS TTM là số
**dẫn xuất** — đếm xem bao nhiêu mã có EPS TTM *y hệt* quý trước:

In [18]:
co_quy_truoc = QUY[QUY["eps"].notna() & QUY.groupby("symbol", observed=True)["eps"].shift(1).notna()]
ty_le = co_quy_truoc.groupby(["year", "quarter"])["eps_khong_doi"].mean().unstack().mul(100).round(0)
print("% số mã có EPS TTM không đổi so với quý trước:")
print(ty_le.loc[ty_le.index >= HOM_NAY.year - 6].to_string())
ky_loi = co_quy_truoc.groupby(["year", "quarter"])["eps_khong_doi"].mean()
ky_loi = ky_loi[ky_loi > 0.5]
print()
print(f"Kỳ có hơn một nửa số mã đứng yên: {[f'Q{q}/{y}' for y, q in ky_loi.index]}")
fpt = QUY[(QUY["symbol"] == "FPT") & (QUY["year"] >= HOM_NAY.year - 1)][["year", "quarter", "eps", "npat"]]
print()
print("Ví dụ FPT — EPS TTM đứng yên trong khi LNST quý vẫn đổi:")
print(fpt.to_string(index=False))

% số mã có EPS TTM không đổi so với quý trước:
quarter     1    2     3     4
year                          
2020      0.0  0.0   0.0   0.0
2021      0.0  0.0   0.0   0.0
2022      0.0  0.0   0.0  84.0
2023      0.0  0.0  84.0   0.0
2024      0.0  0.0   0.0   0.0
2025      0.0  0.0   1.0   0.0
2026     90.0  0.0   0.0   NaN

Kỳ có hơn một nửa số mã đứng yên: ['Q3/2017', 'Q4/2022', 'Q3/2023', 'Q1/2026']

Ví dụ FPT — EPS TTM đứng yên trong khi LNST quý vẫn đổi:
 year  quarter         eps         npat
 2025        1 6699.461058 2.174301e+12
 2025        2 6958.562747 2.257463e+12
 2025        3 6299.285330 2.434841e+12
 2025        4 6589.654590 2.502704e+12
 2026        1 6589.654590 2.487372e+12
 2026        2 6519.935099 2.567587e+12


EPS TTM **đứng yên cả một quý** với phần lớn số mã: 84% ở Q4/2022 và Q3/2023, 90%
ở Q1/2026 — trong khi lợi nhuận sau thuế quý (`npat`) vẫn thay đổi, như FPT ở trên.
Tín hiệu "EPS TTM tăng so với quý trước" sẽ tắt ngấm ở đúng những quý đó với gần
như mọi mã, không lỗi, không cảnh báo.

Nên tín hiệu lợi nhuận dưới đây dùng **`npat` quý so với cùng quý năm trước** (với
`CT`, `npat` quý là riêng quý; so cùng kỳ tự khử mùa vụ). EPS TTM chỉ còn làm mẫu
số P/E — đúng như finlens tính `pe`.

### 6.4 ⚠️ P/E hằng ngày phải tính trên giá thô

P/E hằng ngày = giá hôm nay ÷ EPS TTM đã biết. Nhưng giá nào? Kiểm bằng cách tái
tạo cột `pe` của chính finlens tại ngày cuối quý:

In [19]:
t0 = time.perf_counter()
THO = tai_gia(VU_TRU, nam=7, adjusted=False)
print(f"Giá thô 7 năm: {len(THO):,} dòng · {time.perf_counter() - t0:.0f} giây")

GIA = GIA.merge(THO[["symbol", "date", "close"]].rename(columns={"close": "close_tho"}), on=["symbol", "date"], how="left")
gia_cuoi_quy = GIA[["symbol", "date", "close", "close_tho"]].sort_values("date").rename(columns={"date": "cuoi_quy"})
kt = pd.merge_asof(QUY.dropna(subset=["pe", "eps"]).sort_values("cuoi_quy"), gia_cuoi_quy, on="cuoi_quy", by="symbol").dropna(subset=["close"])

for ten, cot in [("giá điều chỉnh", "close"), ("giá thô", "close_tho")]:
    ty_so = kt["pe"] / (kt[cot] * 1_000 / kt["eps"])
    print(f"pe finlens ÷ P/E tự tính từ {ten:<14}: trung vị {ty_so.median():.4f} · "
          f"khoảng [{ty_so.min():.4f}, {ty_so.max():.4f}] trên {len(ty_so)} quan sát")

Giá thô 7 năm: 164,704 dòng · 38 giây
pe finlens ÷ P/E tự tính từ giá điều chỉnh: trung vị 1.3200 · khoảng [1.0000, 5.3047] trên 2527 quan sát
pe finlens ÷ P/E tự tính từ giá thô       : trung vị 1.0000 · khoảng [1.0000, 1.0000] trên 2527 quan sát


`pe` của finlens = **giá thô** cuối quý ÷ EPS TTM, khớp tuyệt đối. Tính P/E bằng
giá điều chỉnh thì P/E quá khứ bị **kéo thấp** (giá điều chỉnh luôn ≤ giá thô ở
quá khứ) — mã nào chia cổ tức/thưởng nhiều thì trông "rẻ" giả tạo trong backtest.

Và chú ý: cột `pe` của finlens **chính nó đã là nhìn trước** nếu gắn vào ngày cuối
quý — nó dùng EPS mà ngày đó chưa ai biết.

### 6.5 Hai bản line phụ: nhìn trước và có độ trễ

In [20]:
def gan_co_ban(gia_ma: pd.DataFrame, quy_ma: pd.DataFrame, *, nhin_truoc: bool) -> pd.DataFrame:
    """Dời số quý về từng phiên. nhin_truoc=True: dùng từ ngày cuối quý (SAI)."""
    moc = "cuoi_quy" if nhin_truoc else "han_cong_bo"
    q = quy_ma.sort_values(moc)[[moc, "year", "quarter", "eps", "lnst_tang", "roe_yoy"]]
    m = pd.merge_asof(gia_ma.sort_values("date"), q, left_on="date", right_on=moc,
                      allow_exact_matches=nhin_truoc)  # bản trễ: chỉ từ phiên SAU ngày hết hạn
    m["pe_ngay"] = np.where(m["eps"] > 0, m["close_tho"] * 1_000 / m["eps"], np.nan)
    return m


vd = GIA[GIA["symbol"] == MA]
a = gan_co_ban(vd, QUY[QUY["symbol"] == MA], nhin_truoc=True)
b = gan_co_ban(vd, QUY[QUY["symbol"] == MA], nhin_truoc=False)
thay = []
for (y, q), _ in QUY[QUY["symbol"] == MA].groupby(["year", "quarter"]):
    if y < HOM_NAY.year - 1:
        continue
    da = a.loc[(a["year"] == y) & (a["quarter"] == q), "date"].min()
    db = b.loc[(b["year"] == y) & (b["quarter"] == q), "date"].min()
    thay.append({"quý": f"Q{q}/{y}", "bản nhìn trước thấy từ": da, "bản có độ trễ thấy từ": db,
                 "chênh (ngày lịch)": (db - da).days if pd.notna(db) else None})
print(f"{MA} — ngày đầu tiên mỗi bản 'biết' số của quý:")
pd.DataFrame(thay)

HPG — ngày đầu tiên mỗi bản 'biết' số của quý:


,quý,bản nhìn trước thấy từ,bản có độ trễ thấy từ,chênh (ngày lịch)
0,Q1/2025,2025-03-31,2025-05-05,35
1,Q2/2025,2025-06-30,2025-09-03,65
2,Q3/2025,2025-09-30,2025-10-31,31
3,Q4/2025,2025-12-31,2026-04-01,91
4,Q1/2026,2026-03-31,2026-05-04,34
5,Q2/2026,2026-06-30,2026-09-03,65


### 6.6 Nhìn trước đáng bao nhiêu — đo trực tiếp

Trước khi chạy chiến lược, đo thẳng thứ mà bản nhìn trước được tặng: lợi suất
trong **khoảng giữa ngày cuối quý và hạn công bố**, chia theo việc lợi nhuận sau
thuế quý đó cao hơn hay thấp hơn cùng quý năm trước. Lợi suất đã trừ trung bình của mọi mã cùng quý (khử biến động
chung của thị trường). Để so, đo cả **30 ngày sau** hạn công bố — khoảng mà bản có
độ trễ thật sự giao dịch được.

In [21]:
gia_ngay = GIA[["symbol", "date", "close"]].sort_values("date")
ev = QUY.dropna(subset=["lnst_tang_pct"]).copy()
ev["sau_30"] = (ev["han_cong_bo"] + pd.Timedelta(days=30)).astype(kieu_ngay)
for moc, ten in [("cuoi_quy", "p0"), ("han_cong_bo", "p1"), ("sau_30", "p2")]:
    ev = pd.merge_asof(ev.sort_values(moc), gia_ngay.rename(columns={"date": moc, "close": ten}), on=moc, by="symbol")
ev = ev[(ev["cuoi_quy"] >= pd.Timestamp(lui_ngay(HOM_NAY, nam=6))) & (ev["sau_30"] <= GIA["date"].max())]
ev = ev.dropna(subset=["p0", "p1", "p2"])

ev["truoc_cong_bo"] = ev["p1"] / ev["p0"] - 1
ev["sau_cong_bo"] = ev["p2"] / ev["p1"] - 1
for c in ["truoc_cong_bo", "sau_cong_bo"]:
    ev[c] = ev[c] - ev.groupby(["year", "quarter"])[c].transform("mean")  # vượt trung bình cùng quý
ev["nhom"] = pd.qcut(ev["lnst_tang_pct"].rank(method="first"), 5,
                     labels=["1 · LNST giảm mạnh nhất", "2", "3", "4", "5 · LNST tăng mạnh nhất"])

print(f"{len(ev):,} sự kiện (mã × quý) · {ev['symbol'].nunique()} mã · {ev['cuoi_quy'].min():%m/%Y} → {ev['cuoi_quy'].max():%m/%Y}\n")
for c, ten in [("truoc_cong_bo", "cuối quý → hạn công bố"), ("sau_cong_bo", "30 ngày sau hạn công bố")]:
    tang, giam = ev.loc[ev["lnst_tang"] > 0, c], ev.loc[ev["lnst_tang"] <= 0, c]
    chenh = tang.mean() - giam.mean()
    t = chenh / math.sqrt(tang.var() / len(tang) + giam.var() / len(giam))
    print(f"{ten:<26} LNST tăng − LNST giảm = {chenh * 100:+.2f} điểm % (t = {t:.2f})")

(ev.groupby("nhom", observed=True)[["truoc_cong_bo", "sau_cong_bo"]].mean() * 100).round(2).rename(
    columns={"truoc_cong_bo": "cuối quý → hạn công bố, %", "sau_cong_bo": "30 ngày sau hạn, %"}
)

2,163 sự kiện (mã × quý) · 99 mã · 12/2020 → 06/2026

cuối quý → hạn công bố     LNST tăng − LNST giảm = +3.96 điểm % (t = 6.41)
30 ngày sau hạn công bố    LNST tăng − LNST giảm = +1.53 điểm % (t = 3.34)


,"cuối quý → hạn công bố, %","30 ngày sau hạn, %"
nhom,,
1 · LNST giảm mạnh nhất,-2.87,-0.35
2,-1.79,-1.47
3,0.72,0.50
4,1.32,-0.31
5 · LNST tăng mạnh nhất,2.62,1.63


In [22]:
theo_quy = ev.groupby("quarter").apply(
    lambda d: (d.loc[d["lnst_tang"] > 0, "truoc_cong_bo"].mean() - d.loc[d["lnst_tang"] <= 0, "truoc_cong_bo"].mean()) * 100
)
print("Chênh theo quý, điểm %:", theo_quy.round(2).to_dict())
thanh_doi_mau(
    pd.DataFrame({"quy": [f"Q{q} · trễ {DO_TRE[q]} ngày" for q in theo_quy.index], "chenh": theo_quy.values}),
    x="quy",
    y="chenh",
    tieu_de="Phần thưởng của việc nhìn trước, theo quý",
    phu_de="Lợi suất vượt trung bình (LNST tăng − LNST giảm so với cùng kỳ) trong khoảng cuối quý → hạn công bố",
    nhan_y="điểm %",
    dinh_dang_nhan="{:+.2f}",
)

Chênh theo quý, điểm %: {1: 2.51, 2: 4.58, 3: 2.73, 4: 5.79}


Trên 2.163 sự kiện của 99 mã, trong khoảng **cuối quý → hạn công bố** nhóm LNST tăng
vượt nhóm LNST giảm **+3,96 điểm %** (t = 6,41), tăng đều từ nhóm 1 (−2,87%) tới
nhóm 5 (+2,62%). Đó chính là thứ bản nhìn trước được tặng: nó vào lệnh **trước** khi
thị trường biết. Phần thưởng lớn nhất ở Q4 (+5,79 điểm, cửa sổ 90 ngày) và Q2
(+4,58, cửa sổ 60 ngày) — cửa sổ càng dài, giá càng có nhiều thời gian phản ánh
thông tin mà bản nhìn trước biết sẵn.

**30 ngày sau** hạn công bố vẫn còn chênh +1,53 điểm (t = 3,34): một phần phản ứng
chậm mà bản có độ trễ **vẫn** giao dịch được một cách hợp lệ. Hai con số này tách
lợi thế thật khỏi lợi thế do nhìn trước.

### 6.7 Cùng một chiến lược, hai bản line phụ

Luật viết ra trước khi chạy: **giữ cổ phiếu khi** lợi nhuận sau thuế quý mới nhất
cao hơn cùng quý năm trước **và** ROE quý cao hơn cùng quý năm trước **và** P/E hằng
ngày (giá thô ÷ EPS TTM) dưới 20. Một tài khoản 100 triệu riêng cho mỗi mã, 6 năm, đủ luật Việt Nam từ
`bt_vn`. Hai bản chỉ khác đúng một chỗ: ngày số quý bắt đầu có mặt.

In [23]:
class CoBan(bt_vn.ChienLuocVN):
    params = (("pe_toi_da", 20.0),)

    def __init__(self):
        super().__init__()
        self.lenh = None

    def notify_order(self, order):
        super().notify_order(order)
        if order.status in (order.Completed, order.Canceled, order.Margin, order.Rejected, order.Expired):
            self.lenh = None

    def dieu_kien(self) -> bool:
        d = self.data  # NaN so sánh luôn False → chưa có số thì đứng ngoài
        return d.lnst_tang[0] > 0 and d.roe_yoy[0] > 0 and d.pe_ngay[0] < self.p.pe_toi_da

    def next(self):
        if self.lenh:
            return
        if not self.getposition().size:
            if self.dieu_kien():
                self.lenh = self.buy()
        elif not self.dieu_kien():
            self.lenh = self.ban()


COT_CO_BAN = ("lnst_tang", "roe_yoy", "pe_ngay")
BAT_DAU = pd.Timestamp(lui_ngay(HOM_NAY, nam=6))
MA_CO_BAN = sorted(set(QUY["symbol"]) & set(GIA["symbol"]))

t0 = time.perf_counter()
hang, ls = [], {"nhìn trước": {}, "có độ trễ": {}}
for ma in MA_CO_BAN:
    g, q = GIA[GIA["symbol"] == ma], QUY[QUY["symbol"] == ma]
    for ban, nt in [("nhìn trước", True), ("có độ trễ", False)]:
        f = gan_co_ban(g, q, nhin_truoc=nt)
        f = f[f["date"] >= BAT_DAU]
        if len(f) < 250:
            continue
        r = chay(f, CoBan, COT_CO_BAN)
        ls[ban][ma] = r["loi_suat"]
        hang.append({"mã": ma, "bản": ban, "cagr": r["tom_tat"]["cagr"], "so_vong": r["tom_tat"]["so_vong_dong"]})
print(f"{len(hang)} lần chạy backtrader · {time.perf_counter() - t0:.0f} giây")

kq_cb = pd.DataFrame(hang).pivot(index="mã", columns="bản", values="cagr").dropna()
kq_cb["chênh"] = kq_cb["nhìn trước"] - kq_cb["có độ trễ"]
print(f"{len(kq_cb)} mã đủ dữ liệu cho cả hai bản")
print(f"Chênh CAGR (nhìn trước − có độ trễ), điểm %: trung bình {kq_cb['chênh'].mean() * 100:+.2f} · "
      f"trung vị {kq_cb['chênh'].median() * 100:+.2f} · |chênh| trung bình {kq_cb['chênh'].abs().mean() * 100:.2f}")
print(f"Bản nhìn trước tốt hơn ở {(kq_cb['chênh'] > 0).sum()}/{len(kq_cb)} mã · "
      f"tệ hơn ở {(kq_cb['chênh'] < 0).sum()} · bằng nhau ở {(kq_cb['chênh'] == 0).sum()}")

196 lần chạy backtrader · 67 giây
98 mã đủ dữ liệu cho cả hai bản
Chênh CAGR (nhìn trước − có độ trễ), điểm %: trung bình +1.88 · trung vị +0.45 · |chênh| trung bình 5.43
Bản nhìn trước tốt hơn ở 52/98 mã · tệ hơn ở 42 · bằng nhau ở 4


In [24]:
ro = {}
for ban in ls:
    m = pd.DataFrame({k: v for k, v in ls[ban].items() if k in kq_cb.index})
    ro[ban] = m.mean(axis=1)  # bình quân lợi suất ngày của các tài khoản ≈ rổ chia đều

tong = []
for ban, s in ro.items():
    t = bt_vn.chi_so_hieu_qua(s)
    tong.append({"bản": ban, "CAGR %": round(t["cagr"] * 100, 2), "biến động %": round(t["bien_dong"] * 100, 2),
                 "sụt sâu nhất %": round(t["mdd"] * 100, 2), "Sharpe (rf=0)": round(t["sharpe"], 3)})
tong.append({"bản": "chênh (nhìn trước − có độ trễ)",
             **{k: round(tong[0][k] - tong[1][k], 3 if "Sharpe" in k else 2) for k in tong[0] if k != "bản"}})
pd.DataFrame(tong)

,bản,CAGR %,biến động %,sụt sâu nhất %,Sharpe (rf=0)
0,nhìn trước,11.14,8.63,-20.52,1.267
1,có độ trễ,9.24,8.49,-19.99,1.084
2,chênh (nhìn trước − có độ trễ),1.90,0.14,-0.53,0.183


In [25]:
duong(
    pd.concat([pd.DataFrame({"date": s.index, "von": (1 + s).cumprod().values * 100, "duong": f"Bản {ban}"}) for ban, s in ro.items()]),
    x="date",
    y="von",
    theo="duong",
    tieu_de=f"Rổ {len(kq_cb)} tài khoản — cùng luật, khác đúng ngày số quý có mặt",
    phu_de="Mỗi mã một tài khoản 100 triệu · bình quân lợi suất ngày · đã trừ phí và thuế bán",
    nhan_y="giá trị (gốc = 100)",
)

In [26]:
fig = go.Figure(go.Histogram(x=kq_cb["chênh"] * 100, nbinsx=40, marker=dict(color=CHUOI[0], line=dict(width=0))))
fig.add_vline(x=0, line_width=1, line_color="#888")
fig.update_layout(
    title_text="Chênh CAGR từng mã: nhìn trước − có độ trễ<br>"
    "<sub style='color:#52514e'>Mỗi cột là số mã · cùng luật, cùng giá, chỉ khác ngày số quý có mặt</sub>",
    xaxis_title="điểm %/năm", yaxis_title="số mã", height=420, showlegend=False, bargap=0.05,
)
fig

Bản nhìn trước cho rổ CAGR **11,14%** so với **9,24%** của bản có độ trễ —
**+1,90 điểm %/năm** chỉ nhờ biết số sớm; Sharpe 1,267 so với 1,084. Từng mã thì
nhiễu hơn nhiều: chênh trung bình +1,88 điểm nhưng trung vị chỉ +0,45, bản nhìn
trước thắng ở 52/98 mã và thua ở 42, |chênh| bình quân 5,43 điểm/năm.

Nghĩa là: nhìn trước không phải lúc nào cũng làm đẹp **một** mã cụ thể — nó dời thời
điểm vào/ra, có khi rơi đúng nhịp xấu — nhưng **về trung bình nó đẩy kết quả lên**,
khớp với phần thưởng đo trực tiếp ở 6.6. Trên một mã đơn lẻ, khoảng lệch ±5 điểm/năm
đủ để biến một chiến lược tầm thường thành một chiến lược "đáng trình bày".

finlens không cho biết số quý là số trên BCTC quý hay số đã soát xét/kiểm toán điều
chỉnh. Khi không có ngày công bố thật, hạn trễ bảo thủ (60/90 ngày cho Q2/Q4) là
cách phòng gần nhất.

## Tổng kết

| Chỗ nối | Cách làm đúng |
|---|---|
| Line phụ có đơn vị giá | `bt_vn` quy về VND theo `df.attrs`; sau `merge` phải truyền `don_vi_cot_phu` tường minh |
| `bt.ind.*` vs `df.finlens.*` | RSI/SMA/EMA khớp; MACD lệch ở warm-up · line phụ tính trên lịch sử dài ≠ `bt.ind` tính trên cửa sổ backtest |
| Dòng tiền khối ngoại | bỏ phiên dở dang ở cả dòng tiền · `merge` làm mất `attrs` → truyền `don_vi_gia` tường minh |
| ROE quý | chỉ so trong cùng loại hình (`CT` riêng quý, `NH/CK/BH` quy năm) |
| Tín hiệu lợi nhuận | `npat` quý so cùng kỳ — EPS TTM có quý đứng yên với gần như mọi mã |
| Chỉ số tài chính nhiều mã | tải theo lô nhỏ, kiểm `attrs['finlens']['warnings']` và đếm mã — `truncated=False` không đủ |
| P/E hằng ngày | **giá thô** ÷ EPS TTM — đúng cách finlens tính `pe` |
| Số BCTC | dùng từ phiên sau **hạn công bố** (30/60/30/90 ngày), không phải ngày cuối quý |

---

Tiếp theo: [`86_danh_muc_nhieu_ma.ipynb`](86_danh_muc_nhieu_ma.ipynb) — nhiều feed trong
một `Cerebro`, tái cơ cấu danh mục, và dựng lại chiến lược đà tăng của notebook `34`.